# Kalshi Market History
Search for goalscorer markets and chart their trade history.

In [ ]:
# Install dependencies (only needed in Colab)
!pip install requests pandas matplotlib --quiet

In [ ]:
import requests
import base64
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.backends import default_backend

ELECTIONS_HOST = 'https://api.elections.kalshi.com'
API_PREFIX     = '/trade-api/v2'

_API_KEY     = None
_PRIVATE_KEY = None


def _load_keys(api_key_file='kalshi-ladder/kalshi-ladder/api-key.txt',
               private_key_file='kalshi-ladder/kalshi-ladder/api-private-key.key'):
    global _API_KEY, _PRIVATE_KEY
    with open(api_key_file) as f:
        _API_KEY = f.read().strip()
    with open(private_key_file, 'rb') as f:
        _PRIVATE_KEY = serialization.load_pem_private_key(f.read(), password=None, backend=default_backend())
    print(f'Keys loaded. API key: {_API_KEY[:8]}...')


def _make_headers(method, full_path):
    ts = str(int(datetime.datetime.now().timestamp() * 1000))
    msg = (ts + method.upper() + full_path).encode('utf-8')
    sig = _PRIVATE_KEY.sign(msg, padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.DIGEST_LENGTH), hashes.SHA256())
    return {
        'KALSHI-ACCESS-KEY': _API_KEY,
        'KALSHI-ACCESS-TIMESTAMP': ts,
        'KALSHI-ACCESS-SIGNATURE': base64.b64encode(sig).decode('utf-8'),
    }


def _get(path, params=None):
    full_path = API_PREFIX + path
    headers = _make_headers('GET', full_path)
    resp = requests.get(ELECTIONS_HOST + full_path, headers=headers, params=params)
    resp.raise_for_status()
    return resp.json()


def _fetch_paginated(path, result_key, params):
    rows = []
    p = dict(params)
    while True:
        data = _get(path, p)
        batch = data.get(result_key, [])
        rows.extend(batch)
        cursor = data.get('cursor')
        if not cursor or not batch:
            break
        p['cursor'] = cursor
    return rows


def fetch_market(ticker):
    """Look up a market, falling back to the historical endpoint for markets
    settled before Kalshi's live-data cutoff (see /historical/cutoff)."""
    try:
        return _get(f'/markets/{ticker}').get('market', {})
    except requests.HTTPError as e:
        if e.response.status_code != 404:
            raise
        return _get(f'/historical/markets/{ticker}').get('market', {})


def search_markets(query, series_ticker=None, status=None, date_from=None, date_to=None):
    """
    date_from / date_to: 'YYYY-MM-DD' strings — filters by close_time server-side.
    """
    params = {'limit': 1000}
    if series_ticker:
        params['series_ticker'] = series_ticker
    if status:
        params['status'] = status
    if date_from:
        params['min_close_ts'] = int(pd.Timestamp(date_from, tz='UTC').timestamp())
    if date_to:
        params['max_close_ts'] = int((pd.Timestamp(date_to, tz='UTC') + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)).timestamp())

    query_lower = query.lower()
    matches = []
    page = 0

    while True:
        data = _get('/markets', params)
        batch = data.get('markets', [])
        page += 1

        for market in batch:
            title    = (market.get('title')    or '').lower()
            subtitle = (market.get('subtitle') or '').lower()
            ticker   = (market.get('ticker')   or '').lower()
            if query_lower in title or query_lower in subtitle or query_lower in ticker:
                matches.append(market)

        cursor = data.get('cursor')
        if not cursor or not batch:
            break
        params['cursor'] = cursor
        print(f'  Scanned {page * 1000} markets...', end='\r')

    return matches


def fetch_all_trades(ticker, min_ts=None, max_ts=None):
    cutoff_data = _get('/historical/cutoff')
    cutoff_raw  = cutoff_data.get('trades_created_ts')
    cutoff_ts   = int(pd.Timestamp(cutoff_raw).timestamp()) if cutoff_raw else None

    params = {'ticker': ticker, 'limit': 1000}
    if min_ts:
        params['min_ts'] = min_ts
    if max_ts:
        params['max_ts'] = max_ts

    trades = []

    hist_params = dict(params)
    if cutoff_ts:
        hist_params['max_ts'] = cutoff_ts
    try:
        batch = _fetch_paginated('/historical/trades', 'trades', hist_params)
        print(f'Historical: {len(batch)} trades')
        trades.extend(batch)
    except requests.HTTPError as e:
        print(f'Historical endpoint: {e.response.status_code} {e.response.text}')

    live_params = dict(params)
    if cutoff_ts:
        live_params['min_ts'] = cutoff_ts
    try:
        batch = _fetch_paginated('/markets/trades', 'trades', live_params)
        print(f'Live: {len(batch)} trades')
        trades.extend(batch)
    except requests.HTTPError as e:
        print(f'Live endpoint: {e.response.status_code} {e.response.text}')

    return trades


def trades_to_df(trades):
    rows = []
    for t in trades:
        rows.append({
            'time':   pd.to_datetime(t['created_time']),
            'price':  float(t['yes_price_dollars']) * 100,
            'volume': float(t['count_fp']),
            'side':   t.get('taker_book_side', ''),
        })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    return df.sort_values('time').reset_index(drop=True)


def plot_market(df, market):
    ticker = market.get('ticker', '') if isinstance(market, dict) else market
    title  = market.get('title', ticker) if isinstance(market, dict) else ticker
    result = market.get('result', '') if isinstance(market, dict) else ''

    game_start = pd.to_datetime(market.get('occurrence_datetime')) if isinstance(market, dict) and market.get('occurrence_datetime') else None
    settled_at = pd.to_datetime(market.get('settlement_ts'))       if isinstance(market, dict) and market.get('settlement_ts')       else None

    result_color = {'yes': '#2ecc71', 'no': '#e74c3c'}.get(result.lower(), 'gray')
    result_label = f'Settled {result.upper()}' if result else ''

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True,
                                    gridspec_kw={'height_ratios': [3, 1]})

    suptitle = title + (f'  →  {result.upper()}' if result else '')
    fig.suptitle(suptitle, fontsize=13, fontweight='bold', color=result_color if result else 'black')
    ax1.set_title(ticker, fontsize=9, color='gray', pad=2)

    ax1.plot(df['time'], df['price'], color='steelblue', linewidth=1.2, zorder=3)
    ax1.set_ylabel('Yes Price (¢)')
    ax1.set_ylim(0, 100)
    ax1.axhline(50, color='gray', linestyle='--', linewidth=0.7, alpha=0.5)
    ax1.grid(True, alpha=0.3)

    bids = df[df['side'] == 'bid']
    asks = df[df['side'] == 'ask']
    ax1.scatter(bids['time'], bids['price'], color='#2ecc71', s=15, alpha=0.5, zorder=4, label='Taker buy')
    ax1.scatter(asks['time'], asks['price'], color='#e74c3c', s=15, alpha=0.5, zorder=4, label='Taker sell')

    if game_start:
        ax1.axvline(game_start, color='orange', linestyle='--', linewidth=1.2, label='Game start')
        ax2.axvline(game_start, color='orange', linestyle='--', linewidth=1.2)
    if settled_at:
        ax1.axvline(settled_at, color=result_color, linestyle='-', linewidth=1.5, label=result_label)
        ax2.axvline(settled_at, color=result_color, linestyle='-', linewidth=1.5)

    ax1.legend(fontsize=8)

    colors = ['#2ecc71' if s == 'bid' else '#e74c3c' for s in df['side']]
    ax2.bar(df['time'], df['volume'], width=pd.Timedelta(seconds=2), color=colors, alpha=0.7)
    ax2.set_ylabel('Volume')
    ax2.set_xlabel('Time (UTC)')
    ax2.grid(True, alpha=0.3)

    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


def print_summary(df, market):
    ticker = market.get('ticker', '') if isinstance(market, dict) else market
    title  = market.get('title', ticker) if isinstance(market, dict) else ticker
    result = market.get('result', '').upper() if isinstance(market, dict) else ''
    vwap   = (df['price'] * df['volume']).sum() / df['volume'].sum()

    print(f'=== {title} ({ticker}) ===')
    if result:
        print(f'Result:        {result}')
    print(f'Trades:        {len(df)}')
    print(f'Total volume:  {df["volume"].sum():.0f} contracts')
    print(f'Time range:    {df["time"].min()} → {df["time"].max()}')
    print(f'Price range:   {df["price"].min():.1f}¢ → {df["price"].max():.1f}¢')
    print(f'Open:          {df["price"].iloc[0]:.1f}¢')
    print(f'Close:         {df["price"].iloc[-1]:.1f}¢')
    print(f'VWAP:          {vwap:.1f}¢')


print('Functions loaded.')

In [ ]:
import os

_KEY_FILE     = '/content/api-key.txt'
_PRIVKEY_FILE = '/content/api-private-key.key'

if not os.path.exists(_KEY_FILE) or not os.path.exists(_PRIVKEY_FILE):
    from google.colab import files
    print('Upload api-key.txt and api-private-key.key when prompted.')
    uploaded = files.upload()
    for name, data in uploaded.items():
        with open(f'/content/{name}', 'wb') as f:
            f.write(data)

_load_keys(api_key_file=_KEY_FILE, private_key_file=_PRIVKEY_FILE)

## Step 0 — Browse series (optional)
Use this to find the right `SERIES` ticker for your search.

In [ ]:
SERIES_KEYWORD = 'world cup'  # search series titles by keyword, or '' to list all

data = _get('/series')
for s in data.get('series', []):
    title = s.get('title') or ''
    if SERIES_KEYWORD.lower() in title.lower():
        print(f"{s['ticker']:<20} {title}")

**World Cup player market series**

| Series | Type |
|---|---|
| `KXWCGOAL` | Goalscorer |
| `KXWCAST` | Assist |
| `KXWCSOA` | Score or Assist |

## Step 1 — Search for a market
Set `SEARCH_QUERY` to a player name, team, or keyword. Optionally narrow by `STATUS` (`open`, `closed`, `settled`) or `SERIES` if you know it.

In [ ]:
SEARCH_QUERY = 'Ronaldo'    # player name, team, or any keyword
STATUS       = 'settled'    # 'open', 'closed', 'settled', or None for all
SERIES       = 'KXWCGOAL'  # e.g. 'KXWCGOAL' to narrow to a specific series, or None
DATE_FROM    = '2026-06-17' # 'YYYY-MM-DD', or None for no lower bound
DATE_TO      = '2026-06-23' # 'YYYY-MM-DD', or None for no upper bound

results = search_markets(SEARCH_QUERY, series_ticker=SERIES, status=STATUS,
                         date_from=DATE_FROM, date_to=DATE_TO)

if not results:
    print(f'No markets found for "{SEARCH_QUERY}".')
else:
    print(f'Found {len(results)} market(s):\n')
    for m in results:
        status     = m.get('status', '').upper()
        close_time = (m.get('close_time') or '')[:10]
        result     = m.get('result', '')
        settled    = f'  → {result}' if result else ''
        print(f'  [{status}] {m["ticker"]}')
        print(f'          {m.get("title", "")} (closes {close_time}){settled}')
        print()

## Step 2 — Chart a market
Copy a ticker from the search results above and paste it into `TICKER`.

In [ ]:
TICKER = 'KXWCGOAL-26JUN23ENGGHA-ENGHKANE9-1'  # paste ticker here

market = fetch_market(TICKER)
trades = fetch_all_trades(TICKER)

if not trades:
    print('No trades found for this ticker.')
else:
    df = trades_to_df(trades)
    print_summary(df, market)
    plot_market(df, market)

In [ ]:
# ── Activity breakdown ────────────────────────────────────────────────────────
BUCKET     = '5min'            # '15s', '30s', '1min', '5min', '15min', '30min', '1h'
TOP_N      = 5                 # number of most active windows to print
TIMEZONE   = 'America/New_York' # display timezone: 'America/New_York', 'UTC', 'America/Chicago', etc.
GAME_START = '2026-06-23 11:00' # kickoff in TIMEZONE, or None
ZOOM_FROM  = None               # e.g. '2026-06-23 10:30', or None to auto-trim to first trade
ZOOM_TO    = None               # e.g. '2026-06-23 14:00', or None to auto-trim to last trade

df['notional'] = df['price'] / 100 * df['volume']

def parse_ts(s):
    """Parse a timestamp string, localising to TIMEZONE if no tz info present."""
    if s is None:
        return None
    ts = pd.Timestamp(s)
    if ts.tzinfo is None:
        ts = ts.tz_localize(TIMEZONE)
    return ts.tz_convert(TIMEZONE)

def to_tz(s):
    """Convert a UTC-assumed string or Timestamp to display timezone."""
    if s is None:
        return None
    ts = pd.Timestamp(s)
    if ts.tzinfo is None:
        ts = ts.tz_localize('UTC')
    return ts.tz_convert(TIMEZONE)

# Convert trade times to display timezone
df_plot = df.copy()
df_plot['time'] = (df_plot['time'].dt.tz_localize('UTC')
                   if df_plot['time'].dt.tz is None
                   else df_plot['time']).dt.tz_convert(TIMEZONE)

activity = (
    df_plot.set_index('time')
    .resample(BUCKET)
    .agg(trades=('price', 'count'), volume=('volume', 'sum'),
         notional=('notional', 'sum'), avg_price=('price', 'mean'))
    .dropna(subset=['avg_price'])
)

tz_abbr = df_plot['time'].iloc[0].strftime('%Z')

# Top N most active windows
print(f'=== Top {TOP_N} most active {BUCKET} windows ({tz_abbr}) ===')
for ts, row in activity.nlargest(TOP_N, 'trades').iterrows():
    print(f"  {ts.strftime('%H:%M:%S')}  {int(row['trades']):>4} trades  "
          f"{row['volume']:>8.1f} contracts  "
          f"${row['notional']:>9.2f}  avg {row['avg_price']:.1f}¢")

# Resolve display-timezone timestamps
game_start = parse_ts(GAME_START) if GAME_START else to_tz(market.get('occurrence_datetime'))
settled_at = to_tz(market.get('settlement_ts'))
early_close = not GAME_START and settled_at and game_start and settled_at < game_start

# UTC versions for phase filtering against df (which is in UTC)
game_start_utc = game_start.tz_convert('UTC') if game_start else None
settled_utc    = settled_at.tz_convert('UTC')  if settled_at else None
df_utc = df.copy()
if df_utc['time'].dt.tz is None:
    df_utc['time'] = df_utc['time'].dt.tz_localize('UTC')

print(f'\n=== Trades by phase ===')
if early_close:
    print(f'  (Market closed early at {settled_at.strftime("%H:%M")} {tz_abbr}'
          f' — scheduled match time was {game_start.strftime("%H:%M")} {tz_abbr})')
    print(f'  Tip: set GAME_START to the actual kickoff time to see pre/in/post-game splits.')
    phases = [('Pre-settlement',  df_utc[df_utc['time'] <= settled_utc]),
              ('Post-settlement', df_utc[df_utc['time'] >  settled_utc])]
elif game_start_utc:
    pre  = df_utc[df_utc['time'] < game_start_utc]
    live = df_utc[(df_utc['time'] >= game_start_utc) & (df_utc['time'] <= settled_utc)] if settled_utc else df_utc[df_utc['time'] >= game_start_utc]
    post = df_utc[df_utc['time'] > settled_utc] if settled_utc else pd.DataFrame()
    phases = [('Pre-game', pre), ('In-game', live), ('Post-settlement', post)]
else:
    phases = [('All trades', df_utc)]

for label, chunk in phases:
    if chunk.empty:
        continue
    pct = len(chunk) / len(df) * 100
    print(f"  {label:<18} {len(chunk):>5} trades ({pct:>3.0f}%)  "
          f"{chunk['volume'].sum():>10.1f} contracts  "
          f"${chunk['notional'].sum():>10.2f}")

# X-axis bounds
x_min = parse_ts(ZOOM_FROM) if ZOOM_FROM else df_plot['time'].min() - pd.Timedelta(BUCKET)
x_max = parse_ts(ZOOM_TO)   if ZOOM_TO   else df_plot['time'].max() + pd.Timedelta(BUCKET)

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
fig.suptitle('Trade activity over time', fontsize=12)

bar_width = pd.Timedelta(BUCKET) * 0.8
ax1.bar(activity.index, activity['trades'],   width=bar_width, color='steelblue',    alpha=0.8)
ax2.bar(activity.index, activity['notional'], width=bar_width, color='mediumpurple', alpha=0.8)
ax1.set_ylabel(f'Trades per {BUCKET}')
ax2.set_ylabel(f'$ traded per {BUCKET}')
ax2.set_xlabel(f'Time ({tz_abbr})')
ax1.grid(True, alpha=0.3)
ax2.grid(True, alpha=0.3)
ax1.set_xlim(x_min, x_max)

result       = market.get('result', '')
result_color = {'yes': '#2ecc71', 'no': '#e74c3c'}.get(result.lower(), 'gray')
handles      = []
for ax in (ax1, ax2):
    if game_start:
        ax.axvline(game_start, color='orange', linestyle='--', linewidth=1.2)
    if settled_at:
        ax.axvline(settled_at, color=result_color, linestyle='-', linewidth=1.5)
if game_start:
    handles.append(plt.Line2D([0], [0], color='orange', linestyle='--', label='Game start'))
if settled_at:
    handles.append(plt.Line2D([0], [0], color=result_color, linestyle='-',
                               label=f'Settled {result.upper()}'))
if handles:
    ax1.legend(handles=handles, fontsize=8)

fmt = '%H:%M:%S' if pd.Timedelta(BUCKET) < pd.Timedelta('1min') else '%H:%M'
ax1.xaxis.set_major_formatter(mdates.DateFormatter(fmt))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
# ── Trade table ───────────────────────────────────────────────────────────────
# Filter by time. Set to None to see all trades.
TIME_FROM     = None   # e.g. '11:00' or '2026-06-23 11:00' — interpreted in TIMEZONE
TIME_TO       = None   # e.g. '11:15' or '2026-06-23 11:15' — interpreted in TIMEZONE

# VWAP up to this time (printed above the table). None = all visible rows.
VWAP_UNTIL    = None   # e.g. '11:30' or '2026-06-23 11:30' — interpreted in TIMEZONE

# Add a running VWAP column to the table (VWAP up to each row).
SHOW_VWAP_COL = True

# How many rows to show. Set to None to show every row.
MAX_ROWS      = 100

# ── (nothing to change below) ─────────────────────────────────────────────────
_tz = globals().get('TIMEZONE', 'UTC')

_t0 = df['time'].iloc[0]
if _t0.tzinfo is None:
    _t0 = _t0.tz_localize('UTC')
_date_ref = _t0.tz_convert(_tz).date()

def _parse_time_filter(s):
    if s is None:
        return None
    ts = pd.Timestamp(s)
    if ts.year == 1970:
        ts = pd.Timestamp(f'{_date_ref} {s}')
    if ts.tzinfo is None:
        ts = ts.tz_localize(_tz)
    return ts.tz_convert('UTC')

t_from     = _parse_time_filter(TIME_FROM)
t_to       = _parse_time_filter(TIME_TO)
vwap_until = _parse_time_filter(VWAP_UNTIL)

view = df.copy()
if view['time'].dt.tz is None:
    view['time'] = view['time'].dt.tz_localize('UTC')
else:
    view['time'] = view['time'].dt.tz_convert('UTC')
if t_from is not None:
    view = view[view['time'] >= t_from]
if t_to is not None:
    view = view[view['time'] <= t_to]

# Running VWAP column (cumulative over all filtered rows, in time order)
if SHOW_VWAP_COL:
    view = view.copy()
    view['_vwap'] = (view['price'] * view['volume']).cumsum() / view['volume'].cumsum()

# Printed VWAP summary
_vwap_view = view[view['time'] <= vwap_until] if vwap_until is not None else view
if not _vwap_view.empty:
    _vwap = (_vwap_view['price'] * _vwap_view['volume']).sum() / _vwap_view['volume'].sum()
    _vwap_label = f' until {VWAP_UNTIL}' if VWAP_UNTIL else ''
    print(f'VWAP{_vwap_label}: {_vwap:.1f}¢  '
          f'({len(_vwap_view)} trades, {_vwap_view["volume"].sum():.0f} contracts)')

view['time'] = view['time'].dt.tz_convert(_tz)

_cols = ['time', 'price', 'volume', 'side'] + (['_vwap'] if SHOW_VWAP_COL else [])
display_df = view[_cols].copy()
_tz_abbr = display_df['time'].iloc[0].strftime('%Z') if len(display_df) else _tz
display_df['time']   = display_df['time'].dt.strftime('%Y-%m-%d %H:%M:%S.%f').str[:-3]
display_df['price']  = display_df['price'].map('{:.1f}¢'.format)
display_df['volume'] = display_df['volume'].map('{:.2f}'.format)
if SHOW_VWAP_COL:
    display_df['_vwap'] = display_df['_vwap'].map('{:.1f}¢'.format)
_col_names = [f'Timestamp ({_tz_abbr})', 'Yes Price', 'Volume', 'Side'] + (['VWAP'] if SHOW_VWAP_COL else [])
display_df.columns = _col_names

_slice = display_df.iloc[:MAX_ROWS] if MAX_ROWS else display_df
print(f'Showing {len(_slice)} of {len(display_df)} rows'
      + (f' (filtered from {len(df)} total)' if len(display_df) != len(df) else ''))

display(_slice)